### Preapre Training, Validation and Test dataset

In [99]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

from pathlib import Path
import pandas as pd

data_folder = Path("../data")
data_folder.mkdir(parents=True, exist_ok=True)

In [100]:
data_path = "../data/airfoil_self_noise-dataset.xlsx"
data = pd.read_excel(data_path)

print(data.head())

   frequency  attack-angle  chord-length  free-stream-velocity  \
0        800           0.0        0.3048                  71.3   
1       1000           0.0        0.3048                  71.3   
2       1250           0.0        0.3048                  71.3   
3       1600           0.0        0.3048                  71.3   
4       2000           0.0        0.3048                  71.3   

   suction-side-displacement-thickness  scaled-sound-pressure  
0                             0.002663                126.201  
1                             0.002663                125.201  
2                             0.002663                125.951  
3                             0.002663                127.591  
4                             0.002663                127.461  


In [101]:
target_column = "scaled-sound-pressure"

# Input features
x = data.drop(columns=[target_column])

# Target variable
y = data[target_column]

print("Feature columns:")
print(x.columns)

print("\nFeature shape:", x.shape)
print("Target shape:", y.shape)

Feature columns:
Index(['frequency', 'attack-angle', 'chord-length', 'free-stream-velocity',
       'suction-side-displacement-thickness'],
      dtype='object')

Feature shape: (1503, 5)
Target shape: (1503,)


In [102]:
x_train, x_remaining, y_train, y_remaining = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42
)

# Divide the remaining data equally:
# 15% validation and 15% test.

x_validation, x_test, y_validation, y_test = train_test_split(
    x_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42
)

print("Training rows:", len(x_train))
print("Validation rows:", len(x_validation))
print("Test rows:", len(x_test))

Training rows: 1052
Validation rows: 225
Test rows: 226


### Feature scaling

In [103]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_validation_scaled = scaler.transform(x_validation)
x_test_scaled = scaler.transform(x_test)

x_train_scaled = pd.DataFrame(
    x_train_scaled,
    columns=x.columns
)

x_validation_scaled = pd.DataFrame(
    x_validation_scaled,
    columns=x.columns
)

x_test_scaled = pd.DataFrame(
    x_test_scaled,
    columns=x.columns
)

print(x_train_scaled.head())

   frequency  attack-angle  chord-length  free-stream-velocity  \
0  -0.139972      1.834695     -1.207458             -0.751624   
1  -0.139972      0.007693     -0.394587              1.281293   
2   0.317593     -1.136317     -0.394587             -0.751624   
3  -0.710403     -0.675298      0.147326             -0.751624   
4   3.978115     -1.136317      0.960197              1.281293   

   suction-side-displacement-thickness  
0                             0.478222  
1                            -0.475779  
2                            -0.730397  
3                            -0.616516  
4                            -0.678229  


In [104]:
print("Mean after scaling:")
print(x_train_scaled.mean().round(2))

print("\nStandard deviation after scaling:")
print(x_train_scaled.std().round(2))

Mean after scaling:
frequency                             -0.0
attack-angle                          -0.0
chord-length                           0.0
free-stream-velocity                   0.0
suction-side-displacement-thickness   -0.0
dtype: float64

Standard deviation after scaling:
frequency                              1.0
attack-angle                           1.0
chord-length                           1.0
free-stream-velocity                   1.0
suction-side-displacement-thickness    1.0
dtype: float64


In [105]:
target_scaler = StandardScaler()

# Scale the training target
y_train_scaled = target_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1))

# Scale validation and test targets using the same scaler
y_validation_scaled = target_scaler.transform(y_validation.to_numpy().reshape(-1, 1))

y_test_scaled = target_scaler.transform(y_test.to_numpy().reshape(-1, 1))

In [106]:
print("Target mean after scaling:", y_train_scaled.mean())
print("Target standard deviation after scaling:", y_train_scaled.std())

Target mean after scaling: -3.2487742956331577e-15
Target standard deviation after scaling: 1.0


In [107]:
# saving scalers

joblib.dump(target_scaler,"../data/target_scaler.joblib")

joblib.dump(scaler,"../data/feature_scaler.joblib")


['../data/feature_scaler.joblib']

In [108]:
scaled_column_names = [
    "frequency_standardized",
    "attack-angle_standardized",
    "chord-length_standardized",
    "free-stream-velocity_standardized",
    "suction-side-displacement-thickness_standardized"]

In [109]:

train_original = x_train.reset_index(drop=True).copy()

train_original["scaled-sound-pressure"] = (y_train.reset_index(drop=True))

# Standardized training features
train_standardized = x_train_scaled.reset_index(drop=True).copy()
train_standardized.columns = scaled_column_names

# Add the standardized target
train_standardized["target_standardized"] = (y_train_scaled.flatten())

# Put original and standardized columns together
train_final = pd.concat([train_original, train_standardized],axis=1)

train_final.to_csv(data_folder / "train.csv",index=False)

print("Training data saved:", train_final.shape)

Training data saved: (1052, 12)


In [110]:
# Original validation features
validation_original = x_validation.reset_index(drop=True).copy()

# Add the original target in dB
validation_original["scaled-sound-pressure"] = (y_validation.reset_index(drop=True))

# Standardized validation features
validation_standardized = (x_validation_scaled.reset_index(drop=True).copy())

validation_standardized.columns = scaled_column_names

# Add the standardized target
validation_standardized["target_standardized"] = (y_validation_scaled.flatten())

# Put original and standardized columns together
validation_final = pd.concat([validation_original, validation_standardized],axis=1)

validation_final.to_csv(data_folder/"validation.csv",index=False)

print("Validation data saved:", validation_final.shape)

Validation data saved: (225, 12)


In [111]:
# Original test features
test_original = x_test.reset_index(drop=True).copy()

# Add the original target in dB
test_original["scaled-sound-pressure"] = (y_test.reset_index(drop=True))

# Standardized test features
test_standardized = x_test_scaled.reset_index(drop=True).copy()
test_standardized.columns = scaled_column_names

# Add the standardized target
test_standardized["target_standardized"] = (y_test_scaled.flatten())

# Put original and standardized columns together
test_final = pd.concat([test_original, test_standardized],axis=1)

test_final.to_csv(data_folder/"test.csv",index=False)

print("Test data saved:", test_final.shape)

Test data saved: (226, 12)


In [112]:
# add noise values to test subset approx. 15% of the dataset - uniform distribution
# appraoches followed: Add noise to the original target in dB.
#Transform the complete noisy target using the existing target scaler.
# Calculate standardized noise 

test_with_anomalies = test_final.copy()

test_with_anomalies["clean_sound_pressure_db"] = test_with_anomalies["scaled-sound-pressure"]

# This column will contain the noisy sensor reading
test_with_anomalies["observed_sound_pressure_db"] = test_with_anomalies["clean_sound_pressure_db"]

# initalize records
test_with_anomalies["injected_noise_db"] = 0.0
test_with_anomalies["is_anomaly"] = 0
test_with_anomalies["anomaly_severity"] = "normal"

np.random.seed(42)

# Select 15% of test rows to add noise 
anomaly_percentage = 0.15
number_of_anomalies = round(len(test_with_anomalies) * anomaly_percentage)
anomaly_indices = np.random.choice(test_with_anomalies.index, size=number_of_anomalies, replace=False)

print("Total test rows:", len(test_with_anomalies))
print("Selected anomaly rows:", number_of_anomalies)

Total test rows: 226
Selected anomaly rows: 34


In [113]:
one_third = number_of_anomalies // 3

low_indices = anomaly_indices[:one_third]
moderate_indices = anomaly_indices[one_third:one_third * 2]
severe_indices = anomaly_indices[one_third * 2:]

print("Low anomalies:", len(low_indices))
print("Moderate anomalies:", len(moderate_indices))
print("Severe anomalies:", len(severe_indices))

Low anomalies: 11
Moderate anomalies: 11
Severe anomalies: 12


In [114]:
# Low spikes: 3 to 5 dB
low_noise = np.random.uniform(low=3, high=5, size=len(low_indices))

# Moderate spikes: 6 to 10 dB
moderate_noise = np.random.uniform(low=6, high=10, size=len(moderate_indices))

# Severe spikes: 12 to 20 dB
severe_noise = np.random.uniform(low=12, high=20, size=len(severe_indices))

# Add low anomalies
test_with_anomalies.loc[low_indices, "injected_noise_db"] = low_noise
test_with_anomalies.loc[low_indices, "is_anomaly"] = 1
test_with_anomalies.loc[low_indices, "anomaly_severity"] = "low"

# Add moderate anomalies
test_with_anomalies.loc[moderate_indices, "injected_noise_db"] = moderate_noise
test_with_anomalies.loc[moderate_indices, "is_anomaly"] = 1
test_with_anomalies.loc[moderate_indices, "anomaly_severity"] = "moderate"

# Add severe anomalies
test_with_anomalies.loc[severe_indices, "injected_noise_db"] = severe_noise
test_with_anomalies.loc[severe_indices, "is_anomaly"] = 1
test_with_anomalies.loc[severe_indices, "anomaly_severity"] = "severe"


# Add noise to the clean target
test_with_anomalies["observed_sound_pressure_db"] = test_with_anomalies["clean_sound_pressure_db"] + test_with_anomalies["injected_noise_db"]

In [115]:
# Standardize the noisy target using the existing target scaler
test_with_anomalies["observed_target_standardized"] = target_scaler.transform(test_with_anomalies[["observed_sound_pressure_db"]].to_numpy()).flatten()

# Calculate the injected noise in standardized units
test_with_anomalies["injected_noise_standardized"] = test_with_anomalies["observed_target_standardized"] - test_with_anomalies["target_standardized"]

# Check the number of records in each anomaly group
print(test_with_anomalies["anomaly_severity"].value_counts())

# Check the noise range for each anomaly group
print(test_with_anomalies.groupby("anomaly_severity")["injected_noise_db"].agg(["count", "min", "mean", "max"]))

# Save the test dataset containing synthetic anomalies
test_with_anomalies.to_csv(data_folder / "test_with_anomalies.csv", index=False)

print("Saved rows:", len(test_with_anomalies))

anomaly_severity
normal      192
severe       12
low          11
moderate     11
Name: count, dtype: int64
                  count        min       mean        max
anomaly_severity                                        
low                  11   3.192353   4.076675   4.881047
moderate             11   6.699820   8.130203   9.928673
normal              192   0.000000   0.000000   0.000000
severe               12  13.509657  17.611154  19.970030
Saved rows: 226
